In [1]:
from flask import Flask, jsonify, request
import json
import pandas as pd

CONNECTION TO DATABASE: POSTGRE

In [2]:
import psycopg2

# Connect to PostgreSQL database
def connect_to_postgres():
    conn = psycopg2.connect(
        host="localhost",       
        database="lombardia_air_quality", 
        user="airdata_user",    
        password="user"
    )
    return conn

In [3]:
def query_data(query, format='dataframe'):
    """
    Esegue una query su PostgreSQL e restituendo DataFrame o JSON (lista di dict). Parametri:
        query (str): Query SQL da eseguire
        format (str): 'dataframe' o 'json'
    """
    try:
        conn = connect_to_postgres()
        if format == 'dataframe':
            df = pd.read_sql_query(query, conn)
            return df
        
        elif format == 'json':
            cur = conn.cursor()
            cur.execute(query)
            columns = [desc[0] for desc in cur.description]
            rows = cur.fetchall()
            return [dict(zip(columns, row)) for row in rows]
        
        else:
            raise ValueError("format deve essere 'dataframe' o 'json'")

    except Exception as e:
        print(f"Errore: {e}")
        return None

    finally:
        if conn:
            conn.close()

TEST DATAFRAME

In [ ]:
#global df_user
#df_user = pd.DataFrame(columns=[ 'username', 'email', 'type', 'password'])
#df_user['password'] = ['admin', 'user1', 'user2']
#df_user['username'] = ['admin', 'user1', 'user2']
#df_user['email'] = ['adm@gg.it', 'u1@gg.it', 'u2@gg.it']
#df_user['type'] = ['admin', 'user', 'user']
#df_user

LOGIN API

In [4]:
# login
GeoAir_app = Flask("GeoAir")

# Do different API calls for login and signin
@GeoAir_app.route('/api/login', methods=['POST'])
def login():
    global df_user
    # query to retrieve user data
    sql_query = "SELECT * FROM users;"
    df_user = query_data(sql_query, format='dataframe')
    print('Dataframe:')
    print(df_user.head(), '\n') 

    #global df_user
    global logged_in
    logged_in = False
    data = request.get_json()
    username = data.get('username')
    password = data.get('password')

    # if the username or password is empty
    if not username or not password:
        return jsonify({"message": "Username and password are required"}), 400
    # if the username or email is incorrect
    elif username not in df_user['username'].values and username not in df_user['email'].values:
            return jsonify({"message": "Invalid username"}), 401
    # if the username is correct
    elif username in df_user['username'].values or username in df_user['email'].values:
        # check if the password is correct
        match = df_user[df_user['username'] == username]
        pw = match['password'].values[0]
        # if the password is incorrect
        if  pw != password:
            return jsonify({"message": "Invalid password"}), 403
        # if the password is correct
        else: 
            logged_in = True
            return jsonify({"message": "Login successful"}), 200
    else: 
        return jsonify({"message": "ERROR"}),500
    
# admin functionality: add new data?? 


SIGNIN API

In [ ]:
# signin

@GeoAir_app.route('/api/signin', methods=['POST'])
def signin():
    
    global df_user
    new_data = request.get_json()
    username = new_data.get('username')
    email = new_data.get('email')
    password = new_data.get('password')

    # if the username or password or email is empty
    if not username or not password or not email:
        return jsonify({"message": "All fields are necessary to be registered"}), 400
    # if the username or email is already taken
    elif username in df_user['username'].values or email in df_user['email'].values:
        return jsonify({"message": "Username already exists"}), 409
    else:   
        #new_user = pd.Series({
        #"username": username,
        #"email": email,
        #"password": password,
        #"type": "user"
        #})
        # NEED A QUERY TO INSERT THE NEW USER INTO THE DATABASE @ GIAN
        query_input = f"INSERT INTO users (username, email, password) VALUES ('{username}', '{email}', '{password}');"
        #df_user = pd.concat([df_user, pd.DataFrame([new_user])], ignore_index=True)
        return jsonify({"message": "Signin successful"}), 200
    
# Run the
    
      

In [ ]:
# retrive Lombardy air quality data for a given date range, pollutants and number of data
@GeoAir_app.route('/api/lombardy/air_quality_data', methods=['GET'])
def get_lombardy_air_quality_data():
    if logged_in == True:
        try:
            num_data = request.get_json()
            if not num_data:
                    number = 1000
                    date_start = #default date
                    date_end = #default date
                    pollutants = ['PM2.5', 'PM10', 'NO2']
            else: 
                # Extract required parameters
                number = num_data.get('number of data')
                date_start = num_data.get('starting date')
                date_end = num_data.get('ending date')
                pollutants = num_data.get('pollutants')

        # NEEDS A DATABASE QUERY to be defined!!!
            data_lombardy = {
                "region": "Lombardy",
                "air_quality_index": 42,
                "pollutants": {
                    "PM2.5": 12,
                    "PM10": 20,
                    "NO2": 15
                }
            }
            return jsonify(data_lombardy), 200
        except Exception as LoggedInError:
            return jsonify({"message": "Logged_in_error"}), 401
    else:
        return jsonify({"message": "You are not logged in"}), 401
   

In [ ]:
# retrive cities air quality data for a given date range, pollutants and number of data
@GeoAir_app.route('/api/cities/air_quality_data', methods=['GET'])
def get_cities_air_quality_data():
    num_data = request.get_json()
    if not num_data:
            number = 1000
            date_start = #default date
            date_end = #default date
            pollutants = ['PM2.5', 'PM10', 'NO2']
    else: 
        # Extract required parameters
        number = num_data.get('number of data')
        date_start = num_data.get('starting date')
        date_end = num_data.get('ending date')
        pollutants = num_data.get('pollutants')

    city = request.args.get('city')

    # NEEDS A DATABASE QUERY to be defined!!!
    data_cities = {
        "cities": ["Milan", "Rome", "Naples"],
        "air_quality_index": 42,
        "pollutants": {
            "PM2.5": 12,
            "PM10": 20,
            "NO2": 15
        }
    }
    return jsonify(data_cities), 200

In [ ]:
# retrive last lombardy air quality data for a given pollutants 
@GeoAir_app.route('/api/lombardy/last_air_quality_data', methods=['GET'])
def get_lombardy_last_air_quality_data():
    pollutants = request.args.get('pollutants')
    if not pollutants:
        pollutants = ['PM2.5', 'PM10', 'NO2']
    
    # NEEDS A DATABASE QUERY to be defined!!!
    last_data_lombardy = {
        "region": "Lombardy",
        "air_quality_index": 42,
        "pollutants": {
            "PM2.5": 12,
            "PM10": 20,
            "NO2": 15
        }
    }
    return jsonify(last_data_lombardy), 200

SERVER START

In [5]:
# start the Flask server
GeoAir_app.run(port=5000)

 * Serving Flask app 'GeoAir'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
C:\Users\acer\AppData\Local\Temp\ipykernel_15092\2227323200.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
127.0.0.1 - - [26/May/2025 05:47:49] "POST /api/login HTTP/1.1" 401 -


Dataframe:
   id      username            email password                 created_at
0   1         admin  admin@admin.com    admin 2025-05-23 17:49:08.163039
1   2  default user  defaut@user.com     user 2025-05-23 17:49:08.163039 



127.0.0.1 - - [26/May/2025 05:48:04] "POST /api/signin HTTP/1.1" 404 -
